### 0. Setup

In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import ensure_runtime
ensure_runtime(ROOT)

PosixPath('/Users/emma/PycharmProject/id2221-labs')

In [2]:
# Initialize Spark session configured with Delta Lake extension
from src.spark import create_spark
from src.lake import GOLD, SILVER, read_delta

# Restart JVM cleanly with Delta JARs included
spark = create_spark("ml-pipeline")

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import (
    Imputer,
    OneHotEncoder,
    SQLTransformer,
    StandardScaler,
    StringIndexer,
    VectorAssembler,
)
from pyspark.ml.regression import GBTRegressor
from pyspark.sql import functions as F
from src.lake import GOLD, SILVER, read_delta

:: loading settings :: url = jar:file:/Users/emma/PycharmProject/id2221-labs/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/emma/.ivy2/cache
The jars for the packages stored in: /Users/emma/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-13319f81-73b7-4124-b09f-8428838328af;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 457ms :: artifacts dl 22ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0  

### 1. Load Data & Split

In [3]:
# Load integrated table from Gold Lake Layer
df_integrated = read_delta(spark, GOLD / "integrated_taxi_trips")

# Remove anamalous records with negative fare amounts, zero trip distances, and unknown pickup boroughs
df_clean = df_integrated.filter(
    (F.col("fare_amount") > 0) & 
    (F.col("trip_distance") > 0.05) & 
    (~F.col("pickup_borough").isin("Unknown", "EWR"))
)

# Chronological Train / Validation / Test Partitioning
train_raw = df_clean.filter("pickup_date < '2024-02-21'")
val_raw = df_clean.filter("pickup_date >= '2024-02-21' AND pickup_date < '2024-03-11'")
test_raw = df_clean.filter("pickup_date >= '2024-03-11'")

print(f"Training dataset record count:   {train_raw.count():,}")
print(f"Validation dataset record count: {val_raw.count():,}")
print(f"Test dataset record count:       {test_raw.count():,}")

26/09/29 18:14:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Training dataset record count:   4,801,899
Validation dataset record count: 2,040,929


Test dataset record count:       2,731,445


### 2. Feature Engineering

In [4]:
def build_feature_pipeline(
    numeric_cols: list[str],
    categorical_cols: list[str],
    target_col: str = "fare_amount",
) -> Pipeline:
    """Constructs a reusable, configurable PySpark ML feature transformation pipeline.
    """

    # Temporal and cyclical feature generation
    sql_transformer = SQLTransformer(
        statement=f"""
        SELECT *,
            CAST(dayofweek(pickup_datetime) AS DOUBLE) AS day_of_week,
            CAST(month(pickup_datetime) AS DOUBLE) AS month,
            CAST(hour(pickup_datetime) AS DOUBLE) AS hour_of_day,
            CASE WHEN dayofweek(pickup_datetime) IN (1, 7) THEN 1.0 ELSE 0.0 END AS is_weekend,
            SIN(2 * 3.141592653589793 * CAST(hour(pickup_datetime) AS DOUBLE) / 24.0) AS sin_hour,
            COS(2 * 3.141592653589793 * CAST(hour(pickup_datetime) AS DOUBLE) / 24.0) AS cos_hour,
            CAST({target_col} AS DOUBLE) AS label
        FROM __THIS__
        """
    )

    # Missing value imputation for numeric features
    imputed_numeric_cols = [f"{col}_imputed" for col in numeric_cols]
    imputer = Imputer(
        inputCols=numeric_cols,
        outputCols=imputed_numeric_cols,
        strategy="median",
    )

    # Categorical indexing and one-hot encoding
    indexed_cols = [f"{col}_indexed" for col in categorical_cols]
    indexer = StringIndexer(
        inputCols=categorical_cols,
        outputCols=indexed_cols,
        handleInvalid="keep",  # Retain unseen category levels during prediction
    )

    encoded_cols = [f"{col}_vec" for col in categorical_cols]
    encoder = OneHotEncoder(
        inputCols=indexed_cols,
        outputCols=encoded_cols,
        dropLast=True,
    )

    # Vector assembly
    all_feature_cols = (
        imputed_numeric_cols
        + encoded_cols
        + ["day_of_week", "month", "is_weekend", "sin_hour", "cos_hour"]
    )
    assembler = VectorAssembler(
        inputCols=all_feature_cols,
        outputCol="unscaled_features",
        handleInvalid="skip",
    )

    # Standardization
    scaler = StandardScaler(
        inputCol="unscaled_features",
        outputCol="features",
        withStd=True,
        withMean=True,
    )

    return Pipeline(
        stages=[
            sql_transformer,
            imputer,
            indexer,
            encoder,
            assembler,
            scaler,
        ]
    )

# Define feature scope
NUMERIC_FEATURES = ["trip_distance", "temperature_c", "wind_speed_ms", "pm25"]
CATEGORICAL_FEATURES = ["pickup_borough", "dropoff_borough"]

# Instantiate feature engineering pipeline
feature_pipeline = build_feature_pipeline(
    numeric_cols=NUMERIC_FEATURES,
    categorical_cols=CATEGORICAL_FEATURES,
    target_col="fare_amount",
)

### 3. Model Training & Evaluation

In [ ]:
from notebooks.evaluate_ml import evaluate_model
from pyspark.ml import Pipeline
from pyspark.ml.regression import GBTRegressor

# Attach estimator to feature pipeline
regressor = GBTRegressor(featuresCol="features", labelCol="label", maxIter=40, maxDepth=6, seed=42)
full_ml_pipeline = Pipeline(stages=[feature_pipeline, regressor])

# Sample 10% of the training data for fast local execution
train_sample = train_raw.sample(withReplacement=False, fraction=0.1, seed=42).cache()

# Fit pipeline model
t0 = time.time()
pipeline_model = full_ml_pipeline.fit(train_sample)
print(f"End-to-End ML Pipeline fit completed in {time.time() - t0:.2f} seconds.")

# Save trained pipeline model to disk
MODEL_SAVE_PATH = str(ROOT / "data" / "models" / "fare_prediction_gbt_pipeline")
pipeline_model.write().overwrite().save(MODEL_SAVE_PATH)
print(f"Trained Pipeline Model saved to: {MODEL_SAVE_PATH}\n")

# Standardized evaluation (filters zero-distance noise, evaluates metrics, shows error breakdown)
evaluate_model(model=pipeline_model, test_df=test_raw)

26/09/29 18:31:28 WARN CacheManager: Asked to cache already cached data.


End-to-End ML Pipeline fit completed in 120.66 seconds.
Trained Pipeline Model saved to: /Users/emma/PycharmProject/id2221-labs/data/models/fare_prediction_gbt_pipeline



Inference completed in 0.99 seconds across 2,731,445 rows.



           MODEL EVALUATION SUMMARY               
Root Mean Squared Error (RMSE): $5.5231
Mean Absolute Error (MAE):     $2.6181
Coefficient of Determination (R²): 0.8954

=== Error Breakdown by Pickup Borough ===


+--------------+----------+-----+-----+
|pickup_borough|trip_count|mae  |rmse |
+--------------+----------+-----+-----+
|Manhattan     |2451617   |2.28 |4.2  |
|Queens        |239487    |5.48 |12.06|
|Brooklyn      |31900     |5.48 |8.89 |
|Bronx         |7602      |5.83 |8.53 |
|N/A           |781       |33.11|51.73|
|Staten Island |58        |18.65|25.53|
+--------------+----------+-----+-----+



DataFrame[taxi_type: string, vendor_id: int, pickup_datetime: timestamp, dropoff_datetime: timestamp, passenger_count: int, trip_distance: double, pickup_location_id: int, pickup_zone: string, pickup_borough: string, dropoff_location_id: int, dropoff_zone: string, dropoff_borough: string, fare_amount: double, tip_amount: double, tolls_amount: double, total_amount: double, temperature_c: double, wind_speed_ms: double, humidity: double, pm25: double, pm25_unit: string, aqi: double, pickup_date: date, pickup_hour: int, day_of_week: double, month: double, hour_of_day: double, is_weekend: decimal(2,1), sin_hour: double, cos_hour: double, label: double, trip_distance_imputed: double, temperature_c_imputed: double, wind_speed_ms_imputed: double, pm25_imputed: double, pickup_borough_indexed: double, dropoff_borough_indexed: double, pickup_borough_vec: vector, dropoff_borough_vec: vector, unscaled_features: vector, features: vector, prediction: double, residual: double, abs_error: double]

### 4. Retraining and inference

In [28]:
# Load persisted model from disk
loaded_pipeline_model = PipelineModel.load(MODEL_SAVE_PATH)

# Perform inference using loaded model
inference_sample = loaded_pipeline_model.transform(val_raw.limit(5)).select(
    "pickup_datetime", "pickup_borough", "dropoff_borough", "trip_distance", "label", "prediction"
)
print("=== Loaded Model Inference Sample ===")
inference_sample.show(truncate=False)

# Sample 10% of the training data for fast local training
new_val_sample = val_raw.sample(withReplacement=False, fraction=0.1, seed=42).cache()

# Re-fit unfitted full pipeline on updated batch to update transformation parameters and model weights
retrained_pipeline_model = full_ml_pipeline.fit(new_val_sample)

retrained_save_path = str(ROOT / "data" / "models" / "fare_prediction_gbt_pipeline_v2")
retrained_pipeline_model.write().overwrite().save(retrained_save_path)
print(f"Retrained Pipeline Model successfully fit and saved to: {retrained_save_path}")

=== Loaded Model Inference Sample ===


+-------------------+--------------+---------------+-------------+-----+------------------+
|pickup_datetime    |pickup_borough|dropoff_borough|trip_distance|label|prediction        |
+-------------------+--------------+---------------+-------------+-----+------------------+
|2024-03-09 00:01:33|Manhattan     |Manhattan      |2.9          |16.3 |16.78164944086126 |
|2024-03-09 00:03:03|Manhattan     |Manhattan      |4.7          |23.3 |22.843698378156315|
|2024-03-09 00:03:06|Queens        |Manhattan      |8.8          |35.9 |39.28227629299604 |
|2024-03-09 00:03:37|Queens        |Manhattan      |3.5          |18.4 |19.371096356385678|
|2024-03-09 00:03:44|Manhattan     |Manhattan      |0.5          |4.4  |6.109742977585631 |
+-------------------+--------------+---------------+-------------+-----+------------------+



26/09/29 18:38:36 WARN CacheManager: Asked to cache already cached data.


Retrained Pipeline Model successfully fit and saved to: /Users/emma/PycharmProject/id2221-labs/data/models/fare_prediction_gbt_pipeline_v2
